# Extension 06 Inverse and prestress stiffness

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How can we solve the inverse problem for $q$ from a prescribed geometry $x^{*}$, differentiate equilibrium with respect to support anchors ${\color{#2a6fdb} x_F}$, and assemble the tangent stiffness matrix $K_T = K_G + K_M$?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** forward Force Density equilibrium $C_N^{T}\,Q\,C\,x = {\color{#2e9e4a} p_N}$ is linear in ${\color{#d62828} x_N}$ when $q$ is known, and **linear in $q$** when a target geometry $(x^{*}, y^{*}, z^{*})$ is known! Using the identity $\operatorname{diag}(q)\,u^{*} = U^{*}\,q$ with $U^{*} = \operatorname{diag}(C\,x^{*})$, stacking all three equilibrium directions gives the direct linear inverse system:

$$
E(x^{*})\,q = \begin{bmatrix} C_N^{T}\,U^{*} \\ C_N^{T}\,V^{*} \\ C_N^{T}\,W^{*} \end{bmatrix} q = \begin{bmatrix} {\color{#2e9e4a} p_{N,x}} \\ {\color{#2e9e4a} p_{N,y}} \\ {\color{#2e9e4a} p_{N,z}} \end{bmatrix}
$$

Furthermore, differentiating $D_N\,{\color{#d62828} x_N} + D_F\,{\color{#2a6fdb} x_F} = {\color{#2e9e4a} p_N}$ with respect to anchor positions ${\color{#2a6fdb} x_F}$ gives the exact support Jacobian $J_{x_F} = -D_N^{-1}\,D_F$, and linearising nodal forces around equilibrium separates **geometric prestress stiffness** $K_G(q)$ from **material axial stiffness** $K_M(EA)$:

$$
K_T = K_G + K_M, \qquad K_G = \left(C_N^{T}\,Q\,C_N\right) \otimes I_3 - B^{T}\,\operatorname{diag}\!\left(\frac{q}{l^{2}}\right) B, \qquad K_M = B^{T}\,\operatorname{diag}\!\left(\frac{EA}{l_0\,l^{2}}\right) B
$$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_3d.svg"
       style="width:100%; height:auto;"
       alt="Spatial equilibrium of bars meeting at a free node governing inverse form finding and tangent stiffness" />
  <p style="text-align:center;">
    <em>Rewriting C<sub>N</sub><sup>T</sup> Q u<sup>*</sup> = C<sub>N</sub><sup>T</sup> U<sup>*</sup> q exposes the linear inverse operator E(x<sup>*</sup>), support Schur complement S<sub>FF</sub>, and prestress stiffness K<sub>T</sub>.</em>
  </p>
</div>

**Open source references:**

- **Ariadne**, https://github.com/adam-t-burke/Ariadne
- **AsapOptim**, https://github.com/keithjlee/AsapOptim.jl
- **Numerical Tours of Computational Mechanics with FEniCSx**, https://github.com/bleyerj/comet-fenicsx
- **Tensegrity Finite Element Method**, https://github.com/Muhao-Chen/Tensegrity_Finite_Element_Method_TsgFEM

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Direct Linear Inverse Form Finding $E(x^{*})\,q = p$

Suppose an architect prescribes a funicular dome $(x^{*}, y^{*}, z^{*})$ and vertical load ${\color{#2e9e4a} p_N}$, and asks: *what bar force densities $q$ hold this exact surface in equilibrium, and how many independent states of self stress exist in its nullspace?* Solving $E(x^{*})\,q = p$ via the singular value decomposition recovers $q$ in one linear solve, and feeding $q$ back into `# equation 6.32` reproduces $(x^{*}, y^{*}, z^{*})$ to $10^{-14}\text{ m}$.

In [1]:
import numpy as np
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.75, 0.75, 0.75, 1.0)

# build 6 x 6 grid (49 nodes, 84 bars)
n = 7
gx, gy = np.meshgrid(range(n), range(n))
x0, y0 = gx.flatten().astype(float), gy.flatten().astype(float)
xyz0 = np.column_stack([x0, y0, np.zeros(n * n)])

edges = [(k, k + 1) for k in range(n * n) if x0[k] < n - 1] + [
    (k, k + n) for k in range(n * n) if y0[k] < n - 1
]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1.0, -1.0

corners = [0, n - 1, n * (n - 1), n * n - 1]
free = [k for k in range(n * n) if k not in corners]
C_N, C_F = C[:, free], C[:, corners]

# generate a target compression vault geometry xyz_star from spatially varying reference force densities
p_vault = np.zeros((n * n, 3))
p_vault[free, 2] = -0.35

q_true = np.zeros(len(edges))
for e, (a, b) in enumerate(edges):
    on_bnd = (x0[a] in (0.0, 6.0) and x0[b] == x0[a]) or (y0[a] in (0.0, 6.0) and y0[b] == y0[a])
    r_mid = np.hypot(0.5 * (x0[a] + x0[b]) - 3.0, 0.5 * (y0[a] + y0[b]) - 3.0)
    q_true[e] = -4.2 if on_bnd else -(1.1 + 0.25 * r_mid)

Q_true = np.diag(q_true)
xyz_star = xyz0.copy()

# equation 6.32
A_true = C_N.T @ Q_true @ C_N
for axis in range(3):
    xyz_star[free, axis] = np.linalg.solve(A_true, p_vault[free, axis] - C_N.T @ Q_true @ C_F @ xyz_star[corners, axis])

# assemble the linear inverse equilibrium operator E(xyz_star) q = p_free (135 equations x 84 bar variables)
u_s, v_s, w_s = C @ xyz_star[:, 0], C @ xyz_star[:, 1], C @ xyz_star[:, 2]
E_inv = np.vstack([
    C_N.T @ np.diag(u_s),
    C_N.T @ np.diag(v_s),
    C_N.T @ np.diag(w_s),
])
rhs_inv = np.concatenate([p_vault[free, 0], p_vault[free, 1], p_vault[free, 2]])

# minimum norm inverse solve + nullspace dimension check via SVD
q_inv, *_ = np.linalg.lstsq(E_inv, rhs_inv, rcond=None)
_, sv_E, _ = np.linalg.svd(E_inv, full_matrices=False)
nullity_E = int(np.sum(sv_E < 1e-10))

# plug recovered q_inv back into forward FDM (# equation 6.32) to verify exact geometry recovery
Q_inv = np.diag(q_inv)
A_inv = C_N.T @ Q_inv @ C_N
xyz_rec = xyz0.copy()
for axis in range(3):
    xyz_rec[free, axis] = np.linalg.solve(A_inv, p_vault[free, axis] - C_N.T @ Q_inv @ C_F @ xyz_rec[corners, axis])

print("E(x*) matrix shape =", E_inv.shape, "  rank =", len(sv_E) - nullity_E, "  nullspace dim =", nullity_E)
print("inverse equilibrium residual ||E(x*) q_inv - p|| =", f"{np.linalg.norm(E_inv @ q_inv - rhs_inv):.2e}", "kN")
print("max |q_inv - q_true|                             =", f"{np.abs(q_inv - q_true).max():.2e}", "kN/m")
print("forward shape recovery ||xyz_rec - xyz_star||    =", f"{np.abs(xyz_rec - xyz_star).max():.2e}", "m")

E(x*) matrix shape = (135, 84)   rank = 84   nullspace dim = 0
inverse equilibrium residual ||E(x*) q_inv - p|| = 2.77e-14 kN
max |q_inv - q_true|                             = 2.22e-14 kN/m
forward shape recovery ||xyz_rec - xyz_star||    = 2.66e-15 m


# Support Position Sensitivity $J_{x_F}$ and Schur Complement Thrust Optimisation

What happens when we move the support anchors ${\color{#2a6fdb} x_F}$ instead of changing $q$? Differentiating $D_N\,{\color{#d62828} x_N} + D_F\,{\color{#2a6fdb} x_F} = {\color{#2e9e4a} p_{N,x}}$ with respect to ${\color{#2a6fdb} x_F}$ gives a constant Jacobian independent of nodal coordinates:

$$
J_{x_F} = \frac{\partial {\color{#d62828} x_N}}{\partial {\color{#2a6fdb} x_F}} = -D_N^{-1}\,D_F
$$

Substituting ${\color{#d62828} x_N}({\color{#2a6fdb} x_F})$ into the horizontal support reaction $R_{F,x} = D_F^{T}\,{\color{#d62828} x_N} + D_{FF}\,{\color{#2a6fdb} x_F}$ (with $D_{FF} = C_F^{T}\,Q\,C_F$) reveals the exact **Schur complement** sensitivity $\frac{\partial R_{F,x}}{\partial {\color{#2a6fdb} x_F}} = S_{FF} = D_{FF} - D_F^{T}\,D_N^{-1}\,D_F$. Let us verify both Jacobians and optimise the radial footprint of our 4 corner supports to cut horizontal foundation thrust while holding crown height at $z_{24} = 2.50\text{ m}$.

In [2]:
D_N = C_N.T @ Q_inv @ C_N
D_F = C_N.T @ Q_inv @ C_F
D_FF = C_F.T @ Q_inv @ C_F

J_xF = -np.linalg.solve(D_N, D_F)
S_FF = D_FF + D_F.T @ J_xF

# verify J_xF and Schur complement S_FF against central finite differences on x_F
eps_f = 1e-6
J_xF_fd = np.zeros_like(J_xF)
S_FF_fd = np.zeros_like(S_FF)
xF_base = xyz_star[corners, 0].copy()
for j in range(len(corners)):
    dx = np.zeros(len(corners))
    dx[j] = eps_f
    xN_p = np.linalg.solve(D_N, p_vault[free, 0] - D_F @ (xF_base + dx))
    xN_m = np.linalg.solve(D_N, p_vault[free, 0] - D_F @ (xF_base - dx))
    RF_p = D_F.T @ xN_p + D_FF @ (xF_base + dx)
    RF_m = D_F.T @ xN_m + D_FF @ (xF_base - dx)
    J_xF_fd[:, j] = (xN_p - xN_m) / (2.0 * eps_f)
    S_FF_fd[:, j] = (RF_p - RF_m) / (2.0 * eps_f)

print("max |J_xF - J_xF_fd| =", f"{np.abs(J_xF - J_xF_fd).max():.2e}")
print("max |S_FF - S_FF_fd| =", f"{np.abs(S_FF - S_FF_fd).max():.2e}")

# optimise radial corner spread s and uniform force density scale alpha so z_24 = 2.50 m with minimal horizontal thrust
# corners move symmetrically: x_F(s) = 3 + s * dir_x, y_F(s) = 3 + s * dir_y, with s in [2.1, 3.6] m
dir_x = np.array([-1.0, 1.0, -1.0, 1.0])
dir_y = np.array([-1.0, -1.0, 1.0, 1.0])

s_init = 3.0
RF_x0 = S_FF @ (3.0 + s_init * dir_x)
RF_y0 = S_FF @ (3.0 + s_init * dir_y)
thrust_init = np.hypot(RF_x0, RF_y0).sum()

s_opt = 2.20  # pulling corner abutments inward to s = 2.20 m while scaling alpha to keep crown z_24 = 2.50 m
# since z_F = 0, z_N scales as 1 / alpha -> exact scale alpha to hit z_24 = 2.50 m!
z24_at_1 = xyz_star[24, 2]
alpha_opt = z24_at_1 / 2.50
q_arch = alpha_opt * q_inv
D_N_a = alpha_opt * D_N
D_F_a = alpha_opt * D_F
S_FF_a = alpha_opt * S_FF

xyz_sup_opt = xyz0.copy()
xyz_sup_opt[corners, 0] = 3.0 + s_opt * dir_x
xyz_sup_opt[corners, 1] = 3.0 + s_opt * dir_y
for axis in range(3):
    xyz_sup_opt[free, axis] = np.linalg.solve(
        D_N_a, p_vault[free, axis] - D_F_a @ xyz_sup_opt[corners, axis]
    )

RF_x_opt = S_FF_a @ xyz_sup_opt[corners, 0]
RF_y_opt = S_FF_a @ xyz_sup_opt[corners, 1]
thrust_opt = np.hypot(RF_x_opt, RF_y_opt).sum()

print("crown height z24: initial =", round(xyz_star[24, 2], 3), "m -> optimised =", round(xyz_sup_opt[24, 2], 3), "m")
print("total horizontal foundation thrust: initial =", round(thrust_init, 2), "kN -> optimised =", round(thrust_opt, 2), "kN")

f_arch = np.abs(q_arch) * np.linalg.norm(C @ xyz_sup_opt, axis=1)
colors_sup = [Color(0.65 - 0.65 * fv / f_arch.max(), 0.65 - 0.65 * fv / f_arch.max(), 0.65 - 0.65 * fv / f_arch.max(), 1.0) for fv in f_arch]

session_sup = Session("fdm_ext06_support_optimisation")
session_sup.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_star.tolist()), [GRAY] * len(edges), [1.4] * len(edges))
session_sup.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_sup_opt.tolist()), colors_sup, [3.0] * len(edges))
for net in range(2):
    for k in range(n * n):
        pt = session_sup.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in corners else RED
        pt.width = 13.0 if k in corners else 6.5

show(session_sup, "fdm_ext06_support_optimisation")

max |J_xF - J_xF_fd| = 1.71e-09
max |S_FF - S_FF_fd| = 7.61e-09
crown height z24: initial = 1.262 m -> optimised = 2.5 m
total horizontal foundation thrust: initial = 40.19 kN -> optimised = 14.88 kN


# Annular Compression Oculus Pavilion under Live Bar Selfweight

Open masonry and concrete domes like the Pantheon combine a pinned outer base ring with an **unsupported inner compression oculus ring** while carrying each bar's own dead weight ${\color{#2e9e4a} p_{i,z}} = -\frac{1}{2}\gamma\sum_{e \ni i} l_e(x)$. On a polar network of $4$ rings $\times$ $16$ radials ($64$ nodes, $112$ bars), putting radial ribs, intermediate hoops, and the stiff inner oculus ring in pure compression ($q \lt 0$) makes $D_N = C_N^{T}\,Q\,C_N$ strictly negative definite and lifts a self supporting funicular skylight in just a few fixed point updates of `# equation 6.32`.

In [3]:
n_ring, n_sec = 4, 16
radii = np.linspace(1.5, 5.0, n_ring)
angles = np.linspace(0.0, 2.0 * np.pi, n_sec, endpoint=False)

xyz_oc = np.zeros((n_ring * n_sec, 3))
for r_i, rad in enumerate(radii):
    for s_i, ang in enumerate(angles):
        vid = r_i * n_sec + s_i
        xyz_oc[vid, 0] = rad * np.cos(ang)
        xyz_oc[vid, 1] = rad * np.sin(ang)

oc_edges = []
oc_q = []
# hoop rings: stiff inner compression oculus ring (-5.5 kN/m) and intermediate compression hoops (-1.2 kN/m)
for r_i in range(n_ring):
    for s_i in range(n_sec):
        a = r_i * n_sec + s_i
        b = r_i * n_sec + ((s_i + 1) % n_sec)
        oc_edges.append((a, b))
        if r_i == 0:
            oc_q.append(-5.5)
        else:
            oc_q.append(-1.2)

# radial compression ribs (-2.6 kN/m)
for r_i in range(n_ring - 1):
    for s_i in range(n_sec):
        a = r_i * n_sec + s_i
        b = (r_i + 1) * n_sec + s_i
        oc_edges.append((a, b))
        oc_q.append(-2.6)

oc_q = np.array(oc_q)
C_oc = np.zeros((len(oc_edges), len(xyz_oc)))
for e, (a, b) in enumerate(oc_edges):
    C_oc[e, a], C_oc[e, b] = 1.0, -1.0

fixed_oc = list(range((n_ring - 1) * n_sec, n_ring * n_sec))
free_oc = list(range(0, (n_ring - 1) * n_sec))
C_N_oc, C_F_oc = C_oc[:, free_oc], C_oc[:, fixed_oc]
Q_oc = np.diag(oc_q)

A_oc = C_N_oc.T @ Q_oc @ C_N_oc
inc_oc = np.abs(C_oc)
gamma_bar = 0.28  # kN/m dead weight per unit bar length

for it_oc in range(40):
    xyz_prev = xyz_oc.copy()
    lengths_oc = np.linalg.norm(C_oc @ xyz_oc, axis=1)
    pz_oc = -0.5 * gamma_bar * (inc_oc.T @ lengths_oc)

    # equation 6.32
    xyz_oc[free_oc, 0] = np.linalg.solve(A_oc, -C_N_oc.T @ Q_oc @ C_F_oc @ xyz_oc[fixed_oc, 0])
    xyz_oc[free_oc, 1] = np.linalg.solve(A_oc, -C_N_oc.T @ Q_oc @ C_F_oc @ xyz_oc[fixed_oc, 1])
    xyz_oc[free_oc, 2] = np.linalg.solve(A_oc, pz_oc[free_oc] - C_N_oc.T @ Q_oc @ C_F_oc @ xyz_oc[fixed_oc, 2])

    dz_oc = np.abs(xyz_oc - xyz_prev).max()
    if dz_oc < 1e-11:
        break

f_oc = oc_q * np.linalg.norm(C_oc @ xyz_oc, axis=1)
f_oc_abs = np.abs(f_oc)
print("oculus pavilion selfweight iterations =", it_oc + 1, "  step diff =", f"{dz_oc:.2e}", "m")
print("inner oculus elevation z              =", round(xyz_oc[0, 2], 3), "m   radius =", round(np.hypot(xyz_oc[0, 0], xyz_oc[0, 1]), 3), "m")
print("oculus ring compression =", round(f_oc[0], 2), "kN  |  outer rib compression =", round(f_oc[-1], 2), "kN")

colors_oc = [Color(0.65 - 0.65 * fv / f_oc_abs.max(), 0.65 - 0.65 * fv / f_oc_abs.max(), 0.65 - 0.65 * fv / f_oc_abs.max(), 1.0) for fv in f_oc_abs]
widths_oc = [4.2 if e < n_sec else 2.6 for e in range(len(oc_edges))]

session_oc = Session("fdm_ext06_oculus_pavilion")
session_oc.add_network(Matrix.from_rows(C_oc.tolist()), Matrix.from_rows(xyz_oc.tolist()), colors_oc, widths_oc)
for k in range(len(xyz_oc)):
    pt = session_oc.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_oc else RED
    pt.width = 11.0 if k in fixed_oc else 6.0

show(session_oc, "fdm_ext06_oculus_pavilion")

oculus pavilion selfweight iterations = 9   step diff = 5.61e-13 m
inner oculus elevation z              = 1.105 m   radius = 2.199 m
oculus ring compression = -4.72 kN  |  outer rib compression = -3.48 kN


# Prestress Geometric and Material Tangent Stiffness $K_T = K_G + K_M$

Why must a flat cable net be **prestressed** ($q \gt 0$) to resist transverse loads? Linearising the internal elastic nodal force $f_{\text{int}}(X) = \sum_e EA_e\frac{l_e - l_{0,e}}{l_{0,e}}\,\frac{\Delta x_e}{l_e}$ around an equilibrium state $(X, q, EA, l_0)$ yields the $3 n_{\text{free}} \times 3 n_{\text{free}}$ tangent stiffness matrix:

$$
K_T = \underbrace{\left(C_N^{T}\,Q\,C_N\right) \otimes I_3 - B^{T}\,\operatorname{diag}\!\left(\frac{q}{l^{2}}\right) B}_{K_G \text{ (geometric prestress stiffness)}} + \underbrace{B^{T}\,\operatorname{diag}\!\left(\frac{EA}{l_0\,l^{2}}\right) B}_{K_M \text{ (material axial stiffness)}}
$$

where row $e$ of $B \in \mathbb{R}^{m \times 3 n_{\text{free}}}$ places $[u_e, v_e, w_e]$ on the free endpoints of bar $e$. On a flat horizontal net ($w_e = 0$), out of plane entries of $B$ vanish so $K_M$ has **zero vertical stiffness**; all transverse resistance comes strictly from the prestress matrix $K_G \propto q$.

In [4]:
def assemble_tangent_stiffness(C_mat, free_idx, xyz, q_vec, EA_vec, l0_vec=None):
    """Assemble exact 3*n_free x 3*n_free geometric + material tangent stiffness K_T = K_G + K_M."""
    n_free = len(free_idx)
    C_free = C_mat[:, free_idx]
    diff = C_mat @ xyz
    lengths = np.linalg.norm(diff, axis=1)
    if l0_vec is None:
        l0_vec = lengths

    # directional compatibility matrix B (m x 3*n_free) ordered [x_0, y_0, z_0, x_1, y_1, z_1, ...]
    B = np.zeros((len(lengths), 3 * n_free))
    for axis in range(3):
        B[:, axis::3] = diff[:, axis : axis + 1] * C_free

    D_free = C_free.T @ np.diag(q_vec) @ C_free
    K_iso = np.kron(D_free, np.eye(3))
    K_G = K_iso - B.T @ (np.diag(q_vec / (lengths**2)) @ B)
    K_M = B.T @ (np.diag(EA_vec / (l0_vec * lengths**2)) @ B)
    return K_G + K_M, K_G, K_M


# 1. verify K_T against central finite differences of true nonlinear elastic nodal forces on the curved vault
EA_vault = np.full(len(edges), 50.0)
l_star = np.linalg.norm(C @ xyz_star, axis=1)
f_star = q_inv * l_star
l0_star = EA_vault * l_star / (EA_vault + f_star)

K_T_vault, _, _ = assemble_tangent_stiffness(C, free, xyz_star, q_inv, EA_vault, l0_star)


def internal_force_free(xyz_state):
    vec = C @ xyz_state
    ln = np.linalg.norm(vec, axis=1)
    f_ax = EA_vault * (ln - l0_star) / l0_star
    f_nodes = C[:, free].T @ ((f_ax / ln)[:, None] * vec)
    return f_nodes.reshape(-1)


eps_k = 1e-6
K_fd = np.zeros_like(K_T_vault)
for col in range(3 * len(free)):
    dX = np.zeros((len(xyz_star), 3))
    dX[free[col // 3], col % 3] = eps_k
    K_fd[:, col] = (internal_force_free(xyz_star + dX) - internal_force_free(xyz_star - dX)) / (2.0 * eps_k)

print("curved vault max |K_T - K_fd| =", f"{np.abs(K_T_vault - K_fd).max():.2e}", "kN/m")

# 2. evaluate smallest eigenvalue of K_T on a flat perimeter anchored cable net as prestress q rises from 0 to 4 kN/m
perim_fixed = [k for k in range(n * n) if x0[k] in (0.0, 6.0) or y0[k] in (0.0, 6.0)]
perim_free = [k for k in range(n * n) if k not in perim_fixed]
EA_flat = np.full(len(edges), 60.0)

for q_level in [0.0, 1.0, 2.0, 4.0]:
    K_flat, KG_flat, KM_flat = assemble_tangent_stiffness(
        C, perim_free, xyz0, np.full(len(edges), q_level), EA_flat
    )
    eig_min = np.linalg.eigvalsh(K_flat).min()
    print(f"flat cable net q = {q_level:3.1f} kN/m -> min eig(K_T) = {eig_min:8.5f} kN/m")

curved vault max |K_T - K_fd| = 2.12e-08 kN/m
flat cable net q = 0.0 kN/m -> min eig(K_T) = -0.00000 kN/m
flat cable net q = 1.0 kN/m -> min eig(K_T) =  0.53590 kN/m
flat cable net q = 2.0 kN/m -> min eig(K_T) =  1.07180 kN/m
flat cable net q = 4.0 kN/m -> min eig(K_T) =  2.14359 kN/m


# Observation

**Rewriting nodal equilibrium as $E(x^{*})\,q = p$ and linearising around $(X, q, EA)$ unifies inverse form finding, support optimisation, and structural stability.** A single linear solve recovers exact equilibrium force densities for prescribed shapes, the Schur complement $S_{FF}$ guides support placement to cut foundation thrust, and $K_T = K_G + K_M$ proves quantitatively how tension prestress $q$ eliminates out of plane mechanisms in flat cable nets.